In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
import json
from src.data_process import GenerateAnomaly, PrepareDf
from src.metric import ProbTimeDetection, FalseRate
from canari import Optimizer
from pytagi import Normalizer as normalizer
from pytagi import metric


In [2]:
from canari import (
    DataProcess,
    Model,
    Optimizer,
    SKF,
    plot_data,
    plot_prediction,
    plot_skf_states,
    plot_states,
)
from canari.component import LocalTrend, LocalAcceleration, LstmNetwork, WhiteNoise

In [3]:
# Data folder 
data_folder = "/Users/vuongdai/GitHub/bm/detrend_data/monthly/"

# Data file
data_file = "ts_monthly_values_standardize.csv"

# Time file
data_file_time = "ts_monthly_datetimes.csv"

sensor_names = pd.read_csv(data_folder + data_file, nrows=0, delimiter=",").columns.tolist()
df = pd.read_csv(data_folder + data_file, skiprows=1, delimiter=",", header=None)
df_time = pd.read_csv(data_folder + data_file_time, skiprows=1, delimiter=",", header=None)

df_dict = PrepareDf(df, df_time)

# Anomaly info.
anomaly_info_file = "../detrend_data/anomaly_info/anomaly_info.json"
with open(anomaly_info_file, "r") as f:
    anomaly_info = json.load(f)

In [4]:
# Generate anomalies
df_with_anomaly = GenerateAnomaly(data=df_dict, anomaly_info=anomaly_info, col=[0,1])

In [5]:
output_col = [0]
df_temp = df.iloc[:,[0]]
data_processor = DataProcess(
    data=df_temp,
    train_split=0.4,
    validation_split=0.1,
    output_col=output_col,
)
train_data, validation_data, test_data, all_data = data_processor.get_splits()

In [6]:
def model_with_parameters(param):
        model = Model(
            LocalTrend(),
            LstmNetwork(
                look_back_len=param["look_back_len"],
                num_features=1,
                num_layer=1,
                num_hidden_unit=50,
                device="cpu",
                manual_seed=1,
                smoother=False,
            ),
            WhiteNoise(std_error=param["sigma_v"]),
        )

        model.auto_initialize_baseline_states(train_data["y"][0:24])
        num_epoch = 50
        for epoch in range(num_epoch):
            mu_validation_preds, std_validation_preds, states = model.lstm_train(
                train_data=train_data,
                validation_data=validation_data,
            )

            mu_validation_preds_unnorm = normalizer.unstandardize(
                mu_validation_preds,
                data_processor.scale_const_mean[data_processor.output_col],
                data_processor.scale_const_std[data_processor.output_col],
            )

            std_validation_preds_unnorm = normalizer.unstandardize_std(
                std_validation_preds,
                data_processor.scale_const_std[data_processor.output_col],
            )

            validation_obs = data_processor.get_data("validation").flatten()
            validation_log_lik = metric.log_likelihood(
                prediction=mu_validation_preds_unnorm,
                observation=validation_obs,
                std=std_validation_preds_unnorm,
            )

            model.early_stopping(
                evaluate_metric=-validation_log_lik,
                current_epoch=epoch,
                max_epoch=num_epoch,
            )
            model.metric_optim = model.early_stop_metric

            if model.stop_training:
                break

        #### Define SKF model with parameters #########
        abnorm_model = Model(
            LocalAcceleration(),
            LstmNetwork(),
            WhiteNoise(),
        )
        skf = SKF(
            norm_model=model,
            abnorm_model=abnorm_model,
            std_transition_error=param["std_transition_error"],
            norm_to_abnorm_prob=param["norm_to_abnorm_prob"],
        )
        skf.save_initial_states()

        skf.filter(data=all_data)
        log_lik_all = np.nanmean(skf.ll_history)
        skf.metric_optim = -log_lik_all

        skf.load_initial_states()

        return skf

In [7]:
param_space = {
    "look_back_len": [10, 24],
    "sigma_v": [1e-3, 2e-1],
    "std_transition_error": [1e-6, 1e-4],
    "norm_to_abnorm_prob": [1e-6, 1e-4],
}
# Define optimizer
model_optimizer = Optimizer(
    model=model_with_parameters,
    param=param_space,
    num_optimization_trial=10,
    mode="min",
)
model_optimizer.optimize()

#  1/10 - Metric: 1.259 - Parameter: {'look_back_len': 12, 'sigma_v': 0.13278747238261185, 'std_transition_error': 2.9822457482468727e-05, 'norm_to_abnorm_prob': 9.617153635539822e-05}
#  2/10 - Metric: 0.548 - Parameter: {'look_back_len': 21, 'sigma_v': 0.09143649672813922, 'std_transition_error': 7.939526457794583e-06, 'norm_to_abnorm_prob': 2.4343417626683345e-05}
#  3/10 - Metric: 0.592 - Parameter: {'look_back_len': 15, 'sigma_v': 0.1208954577260874, 'std_transition_error': 3.952177587295695e-05, 'norm_to_abnorm_prob': 1.4499553444508106e-05}
#  4/10 - Metric: 0.477 - Parameter: {'look_back_len': 24, 'sigma_v': 0.04911246948494103, 'std_transition_error': 3.616525672977259e-05, 'norm_to_abnorm_prob': 1.314175174702999e-05}
#  5/10 - Metric: 0.930 - Parameter: {'look_back_len': 21, 'sigma_v': 0.014320465826938206, 'std_transition_error': 8.71460115366002e-05, 'norm_to_abnorm_prob': 7.931929839049167e-05}
#  6/10 - Metric: 0.433 - Parameter: {'look_back_len': 22, 'sigma_v': 0.116319

2026-05-22 14:07:18,159	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/Users/vuongdai/ray_results/objective_2026-05-22_14-06-47' in 0.0142s.


# 10/10 - Metric: 0.853 - Parameter: {'look_back_len': 14, 'sigma_v': 0.17248951001016719, 'std_transition_error': 5.98054922094985e-05, 'norm_to_abnorm_prob': 2.5291237662961592e-05}
-----
Optimal parameters at trial #7. Best metric: 0.2521. Best print metric: None. Best param: {'look_back_len': 13, 'sigma_v': 0.11264457427060384, 'std_transition_error': 4.3007274114659204e-05, 'norm_to_abnorm_prob': 9.452260611190452e-05}.
-----


In [10]:
param = model_optimizer.get_best_param()
skf_optim = model_with_parameters(param)

In [ ]:
def _anomaly_score(data, detector, threshold):
    """Run anomaly detection on a single pandas Series."""
    _data = {
        "y": data.values,
        "x": [[] for _ in range(len(data))],
        "time": data.index,
    }
    scores, _ = detector.filter(data=_data)
    return pd.DataFrame({"value": scores > threshold}, index=data.index)


def _apply_recursive(data, detector, threshold):
    """Walk a nested dict of any depth; apply detection at the leaf (a Series/DataFrame)."""
    if isinstance(data, (pd.Series, pd.DataFrame)):
        return _anomaly_score(data, detector, threshold)
    return {
        key: _apply_recursive(value, detector, threshold)
        for key, value in data.items()
    }


def detect_anomaly(data, detector, threshold=0.1):
    return _apply_recursive(data, detector, threshold)

In [12]:
# Estimate anomaly score
anomaly_score_synthetic = detect_anomaly(detector=skf_optim, data=df_with_anomaly)
anomaly_score_test_set = detect_anomaly(detector=skf_optim, data=df_dict)

In [14]:
prob_detection, time_to_detection = ProbTimeDetection(anomaly_score_synthetic, anomaly_info)


In [15]:
prob_detection

{0: {'level': {'0.45': 1.0, '0.01': 0.0, '0.1': 1.0},
  'trend': {'0.01': 1.0, '0.05': 1.0, '0.1': 1.0}},
 1: {'level': {'0.45': 1.0, '0.01': 0.0, '0.1': 1.0},
  'trend': {'0.01': 1.0, '0.05': 1.0, '0.1': 1.0}}}

In [16]:
time_to_detection

{0: {'level': {'0.45': Timedelta('1126 days 00:00:00'),
   '0.01': Timedelta('0 days 00:00:00'),
   '0.1': Timedelta('3026 days 19:12:00')},
  'trend': {'0.01': Timedelta('1372 days 09:36:00'),
   '0.05': Timedelta('1098 days 14:24:00'),
   '0.1': Timedelta('927 days 04:48:00')}},
 1: {'level': {'0.45': Timedelta('1168 days 21:36:00'),
   '0.01': Timedelta('0 days 00:00:00'),
   '0.1': Timedelta('2629 days 19:12:00')},
  'trend': {'0.01': Timedelta('1342 days 02:24:00'),
   '0.05': Timedelta('991 days 04:48:00'),
   '0.1': Timedelta('885 days 00:00:00')}}}

In [17]:
false_rate = FalseRate(anomaly_score_test_set)

In [18]:
false_rate

{0: 0.0,
 1: 0.0,
 2: 0.22648821827201324,
 3: 0.46339761481857394,
 4: 0.0,
 5: 0.0,
 6: 0.046339761481857394,
 7: 0.0,
 8: 0.22648821827201324,
 9: 0.13901928444557218,
 10: 0.4170578533367165,
 11: 0.395131845841785,
 12: 0.0,
 13: 0.0,
 14: 0.5097373763004314,
 15: 0.0,
 16: 0.0,
 17: 0.0,
 18: 0.0,
 19: 0.4170578533367165,
 20: 0.46339761481857394,
 21: 0.7214490931944694,
 22: 0.0,
 23: 0.0,
 24: 0.0,
 25: 0.0,
 26: 0.0,
 27: 0.0,
 28: 0.0,
 29: 0.14039976936382856,
 30: 0.0,
 31: 0.9141186071817192,
 32: 0.04000109517029898,
 33: 2.2256800870511424,
 34: 0.0,
 35: 0.0,
 36: 1.3972209720972097,
 37: 0.0,
 38: 0.0,
 39: 0.0,
 40: 0.0,
 41: 0.0,
 42: 0.20224252491694353,
 43: 0.0,
 44: 0.0,
 45: 0.09837058981955293,
 46: 0.0,
 47: 0.0,
 48: 0.0,
 49: 0.5246902495959778,
 50: 0.0,
 51: 0.0,
 52: 0.0,
 53: 0.34574036511156186,
 54: 0.0,
 55: 0.0,
 56: 0.0,
 57: 0.0,
 58: 0.0,
 59: 0.0,
 60: 0.48993963782696176,
 61: 0.0,
 62: 0.0,
 63: 0.12697722927168434,
 64: 0.8253519902659482,
 6

In [4]:
import numpy as np
rng = np.random.default_rng(seed=42)
numbers = np.round(rng.random(5000), 8)
# print(numbers)
# As a list
numbers_list = numbers.tolist()

# Save to CSV
np.savetxt("random_numbers.csv", numbers, fmt="%.8f", header="value", comments="")